# Hafta 5 — Dil Modelleme: n-gramlardan Sinirsel Modellere

Dil modelleme görevini biçimselleştiriyor, perpleksiteyi tanımlıyor ve sayım tabanlı n-gramlardan Bengio vd. (2003) ileri beslemeli sinirsel dil modeline uzanan mimari evrimi izliyoruz. Bu mimari gömme katmanını ve softmax çıktısını tanıttı ve bu dersteki sonraki her şeyi mümkün kıldı.

## Öğrenme Hedefleri

- Bir dil modelini biçimsel olarak tanımlamak ve olasılığın zincir kuralıyla ilişkilendirmek.
- Perpleksiteyi çapraz entropiden türetmek ve yorumlama ölçeğini açıklamak.
- Kneser–Ney yumuşatmalı bir n-gram dil modeli uygulamak.
- Bengio (2003) sinirsel dil modelini PyTorch ile uygulamak, aynı test akışında n-gram modeliyle karşılaştırmak ve sinirsel modelin üstünlüğünün hangi koşullarda ortaya çıktığını tartışmak.

## Zorunlu Okumalar

- Bengio, Y., Ducharme, R., Vincent, P. ve Jauvin, C. (2003). *A Neural Probabilistic Language Model*.
- Jurafsky ve Martin, Bölüm 3 ve 7.

In [ ]:
import sys, math, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAS_TORCH = True
    torch.manual_seed(0)
except ImportError:
    HAS_TORCH = False
    print("PyTorch bulunamadı — sinirsel dil modeli bölümü atlanacak.")

## 1. Dil modelleme görevi

Bir dil modeli, herhangi bir token dizisine bir $P(w_1, \ldots, w_T)$ olasılığı atar. Zincir kuralıyla,

$$P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_{<t}).$$

Modelleme problemi $P(w_t \mid w_{<t})$'yi tahmin etmektir. Modern NLP'deki her şey — çeviri, özetleme, diyalog, kod üretimi — bu tek problemin akış aşağısındadır.

## 2. Perpleksite

$p$'den çekilen metin üzerinde bir $q$ modelinin çapraz entropisi şudur:

$$H(p, q) = -\mathbb{E}_{w \sim p} \log q(w).$$

Perpleksite bunun üstelidir:

$$\text{PPL}(q) = \exp(H(p, q)) = \exp\!\left(-\frac{1}{N} \sum_{t=1}^{N} \log q(w_t \mid w_{<t})\right).$$

Yorum: $K$ perpleksitesi, modelin ortalamada $K$ seçenek arasından düzgün dağılımla seçim yapıyormuş kadar kararsız olduğu anlamına gelir. $V$ boyutlu bir sözlük üzerinde düzgün bir modelin $\text{PPL} = V$'dir; kusursuz bir modelin $\text{PPL} = 1$'dir.

**Uyarı.** Perpleksite farklı tokenizer'lar arasında doğrudan karşılaştırılamaz. Tokenizer'lar arası doğru ölçü, bayt başına perpleksitedir (bayt başına bit). Aynı nedenle, iki modeli karşılaştırırken **aynı sözlüğü ve aynı test token akışını** kullanmak gerekir; aşağıda bunu yapıyoruz.

## 3. Kneser–Ney yumuşatmalı n-gram dil modelleri

Bir trigram en çok olabilirlik tahmini şudur:

$$P_{\text{MLE}}(w \mid u, v) = \frac{c(u, v, w)}{c(u, v)}.$$

Sıfır sayımlar sıfır olasılık verir; bu yüzden yumuşatma yaparız. **Değiştirilmiş Kneser–Ney** (Chen ve Goodman, 1999), deneysel olarak en güçlü n-gram yumuşatma yöntemidir:

$$P_{\text{KN}}(w \mid u, v) = \frac{\max(c(u, v, w) - D, 0)}{c(u, v)} + \lambda(u, v) P_{\text{devam}}(w \mid v),$$

burada devam olasılığı $P_{\text{devam}}$, sözcükleri ham sıklıklarına göre değil, *göründükleri farklı bağlamların sayısına* göre ağırlıklandırır — Kneser–Ney'in içgörüsü şudur: *San Francisco* sık geçer ama nadiren yeni bir devamdır (*Francisco* hemen hemen yalnızca *San*'dan sonra gelir).

**Sözlük dışı sözcükler.** Eğitimde görülmemiş sözcükler `<unk>` tokenına eşlenir ve devam olasılığının son düzeyi $1/|V|$'lik düzgün bir tabanla birleştirilir; böylece hiçbir olasılık sıfır olmaz.

In [ ]:
class TrigramLM:
    def __init__(self, discount=0.75):
        self.D = discount

    def fit(self, sentences):
        self.tri = Counter()
        self.bi = Counter()
        self.uni = Counter()
        self.bi_ctx = Counter()    # trigram bağlamları için count(u, v)
        self.uni_ctx = Counter()   # bigram bağlamları için count(v)
        self.follow_3 = defaultdict(set)  # (u, v)'den sonra gelen sözcükler
        self.follow_2 = defaultdict(set)  # v'den sonra gelen sözcükler
        self.precede_1 = defaultdict(set) # w'den önce gelen farklı sözcükler (devam olasılığı için)
        self.distinct_bigrams = 0

        for sent in sentences:
            s = ['<s>', '<s>'] + sent + ['</s>']
            for i in range(len(s)):
                self.uni[s[i]] += 1
                if i >= 1:
                    self.bi[(s[i-1], s[i])] += 1
                    self.uni_ctx[s[i-1]] += 1
                    self.follow_2[s[i-1]].add(s[i])
                    self.precede_1[s[i]].add(s[i-1])
                if i >= 2:
                    self.tri[(s[i-2], s[i-1], s[i])] += 1
                    self.bi_ctx[(s[i-2], s[i-1])] += 1
                    self.follow_3[(s[i-2], s[i-1])].add(s[i])

        self.distinct_bigrams = len(self.bi)
        self.V = set(self.uni.keys()) | {'<unk>'}
        return self

    def _map(self, sent):
        # Eğitimde görülmemiş sözcükleri <unk>'a eşle.
        return [t if t in self.V else '<unk>' for t in sent]

    def p_continuation(self, w):
        # P_devam(w) ∝ |{u : c(u, w) > 0}|, 1/|V|'lik düzgün tabanla birleştirilmiş.
        num = len(self.precede_1.get(w, set()))
        p = num / max(self.distinct_bigrams, 1)
        return p if p > 0 else 1.0 / len(self.V)

    def p_kn_bigram(self, w, v):
        c_vw = self.bi[(v, w)]
        c_v = self.uni_ctx[v]
        if c_v == 0:
            return self.p_continuation(w)
        lam = (self.D * len(self.follow_2[v])) / c_v
        return max(c_vw - self.D, 0) / c_v + lam * self.p_continuation(w)

    def p_kn_trigram(self, w, u, v):
        c_uvw = self.tri[(u, v, w)]
        c_uv = self.bi_ctx[(u, v)]
        if c_uv == 0:
            return self.p_kn_bigram(w, v)
        lam = (self.D * len(self.follow_3[(u, v)])) / c_uv
        return max(c_uvw - self.D, 0) / c_uv + lam * self.p_kn_bigram(w, v)

    def perplexity(self, sentences):
        log_p, n = 0.0, 0
        for sent in sentences:
            s = ['<s>', '<s>'] + self._map(sent) + ['</s>']
            for i in range(2, len(s)):
                p = max(self.p_kn_trigram(s[i], s[i-2], s[i-1]), 1e-12)
                log_p += math.log(p)
                n += 1
        return math.exp(-log_p / max(n, 1))

def tokenize(s):
    return re.findall(r"\w+", s.lower())

# Mütevazı bir eğitim derlemi (üretimde: Penn Treebank, WikiText vb.). Örnekler İngilizcedir.
TRAIN_TEXT = (
    "the cat sat on the mat. the dog sat on the rug. "
    "machine learning models learn from data. deep learning is powerful. "
    "natural language processing handles text. the quick brown fox jumps. "
    "language models predict words. neural networks have many parameters. "
    "transformers process sequences in parallel. attention is all you need. " * 60
)
# Test kümesi: çoğunlukla eğitim sözcüklerinin YENİ dizilişleri (genellemeyi ölçer)
# ve sözlük dışı sözcükler içeren bir cümle (<unk> yönetimini gösterir).
TEST_TEXT = (
    "the dog sat on the mat. the cat sat on the rug. "
    "neural networks learn from data. language models process text. "
    "deep learning models predict words. "
    "the cat chased the mouse. "
)

train_sents = [tokenize(s) for s in TRAIN_TEXT.split('.') if s.strip()]
test_sents  = [tokenize(s) for s in TEST_TEXT.split('.') if s.strip()]

lm = TrigramLM(discount=0.75).fit(train_sents)
print(f"Eğitim perpleksitesi (KN trigram): {lm.perplexity(train_sents):.2f}")
print(f"Test perpleksitesi   (KN trigram): {lm.perplexity(test_sents):.2f}")
print(f"Test cümlelerindeki sözlük dışı sözcükler: {sorted({t for s in test_sents for t in s} - lm.V)}")

## 4. Bengio (2003) sinirsel dil modeli

Mimari: son $n - 1$ tokenı sözlük üzerinde bir softmax'a eşleyen ileri beslemeli bir ağ.

$$\mathbf{h} = \tanh(W [\mathbf{e}_{w_{t-n+1}}; \ldots; \mathbf{e}_{w_{t-1}}] + \mathbf{b}),$$

$$P(w_t \mid w_{<t}) = \text{softmax}(U \mathbf{h} + \mathbf{d}).$$

Bu makaleden hâlâ kullandığımız üç yenilik:

1. **Öğrenilen sözcük gömmeleri.** $\mathbf{e}_w \in \mathbb{R}^m$ — her sözcük için dil modeliyle birlikte öğrenilen yoğun bir vektör.
2. **Softmax çıktısı.** Sözlüğün tamamı üzerinde olasılık.
3. **Uçtan uca gradyan eğitimi.** Tüm parametreler log-olabilirlik üzerinde SGD ile öğrenilir.

Karşılaştırmanın adil olması için Bengio modeli de aynı sözlüğü (`<unk>` dâhil) ve aynı test token akışını kullanır: hiçbir test cümlesi atılmaz.

**Ne beklemeli?** Bengio vd. (2003), Brown derleminde en iyi yumuşatılmış trigram modeline göre yaklaşık %24 daha düşük test perpleksitesi bildirdi. Ancak bu üstünlük, sinirsel modelin benzer sözcükler arasında genelleme yapabildiği *gerçekçi büyüklükte* derlemlerde ortaya çıkar. Buradaki oyuncak derlem yalnızca birkaç cümlenin tekrarından oluşur: n-gram modeli onu neredeyse kusursuz ezberler, sinirsel model ise eğitim boyunca aşırı öğrenir (test kaybının dönemler ilerledikçe arttığına dikkat edin). Bu derlemde Kneser–Ney'in daha düşük test perpleksitesi vermesi beklenen bir sonuçtur — üstünlüğü 4. alıştırmada ölçeği büyüterek kendiniz arayın.

In [ ]:
if HAS_TORCH:
    class BengioLM(nn.Module):
        def __init__(self, vocab_size, emb_dim=32, hidden=64, context=3):
            super().__init__()
            self.context = context
            self.emb = nn.Embedding(vocab_size, emb_dim)
            self.W = nn.Linear(context * emb_dim, hidden)
            self.U = nn.Linear(hidden, vocab_size)

        def forward(self, idx):
            # idx: (yığın, bağlam)
            e = self.emb(idx).reshape(idx.size(0), -1)
            h = torch.tanh(self.W(e))
            return self.U(h)  # logit'ler (yığın, sözlük)

    # Sözlüğü eğitim cümlelerinden kur (<unk> dâhil — n-gram modeliyle aynı sözlük)
    all_tokens = ['<s>', '</s>', '<unk>'] + sorted({t for s in train_sents for t in s})
    tok2id = {t: i for i, t in enumerate(all_tokens)}
    id2tok = {i: t for t, i in tok2id.items()}
    V = len(tok2id)
    CONTEXT = 3

    def make_examples(sents):
        X, Y = [], []
        for s in sents:
            # Sözlük dışı tokenlar atılmaz, <unk>'a eşlenir (n-gram modeliyle aynı token akışı).
            ids = [tok2id['<s>']] * CONTEXT + [tok2id.get(t, tok2id['<unk>']) for t in s] + [tok2id['</s>']]
            for i in range(CONTEXT, len(ids)):
                X.append(ids[i-CONTEXT:i])
                Y.append(ids[i])
        return torch.tensor(X), torch.tensor(Y)

    X_train, Y_train = make_examples(train_sents)
    X_test,  Y_test  = make_examples(test_sents)
    print(f"Eğitim örnekleri: {len(X_train)}, test örnekleri: {len(X_test)}")

    model = BengioLM(V, emb_dim=32, hidden=64, context=CONTEXT)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)

    for epoch in range(200):
        model.train()
        opt.zero_grad()
        logits = model(X_train)
        loss = F.cross_entropy(logits, Y_train)
        loss.backward(); opt.step()
        if (epoch + 1) % 50 == 0:
            with torch.no_grad():
                test_loss = F.cross_entropy(model(X_test), Y_test).item() if len(X_test) > 0 else float('nan')
            print(f"dönem {epoch+1:4d}  eğitim_kaybı={loss.item():.4f}  test_kaybı={test_loss:.4f}")

    # Nihai perpleksiteler
    with torch.no_grad():
        train_ppl = math.exp(F.cross_entropy(model(X_train), Y_train).item())
        test_ppl  = math.exp(F.cross_entropy(model(X_test),  Y_test ).item()) if len(X_test) > 0 else float('nan')
    print(f"\nBengio DM     — eğitim PPL: {train_ppl:.2f},  test PPL: {test_ppl:.2f}")
    print(f"KN trigram    — eğitim PPL: {lm.perplexity(train_sents):.2f},  test PPL: {lm.perplexity(test_sents):.2f}")
    print("\nBu oyuncak derlemde n-gram modeli derlemi ezberlediği için genellikle daha düşük test perpleksitesi verir;\n"
          "test kaybının dönemlerle artması sinirsel modelin aşırı öğrendiğini gösterir.")
    print("Not: <unk> eğitimde hiç görülmediğinden iki model de sözlük dışı sözcüklerde yüksek kayıp öder;\n"
          "gerçek derlemlerde nadir eğitim sözcükleri <unk> ile değiştirilerek bu olasılık öğrenilir.")
else:
    print("Bu bölümü çalıştırmak için PyTorch kurun.")

## 5. Bir dil modelinden örnekleme

Eğitilmiş bir dil modeli, tokenları öz-bağlanımlı (autoregressive) olarak örnekleyerek metin üretebilir. Üç yaygın strateji:

- **Açgözlü.** Her zaman argmax'ı al. Tekrarlı, düşük çeşitlilik.
- **Sıcaklıkla örnekleme.** Softmax'tan önce logit'leri $1/\tau$ ile ölçekle. $\tau < 1$ → daha keskin, $\tau > 1$ → daha düz.
- **Top-k / top-p (nucleus).** Örneklemeden önce desteği kısıtla (Holtzman vd., 2020).

In [ ]:
if HAS_TORCH:
    def sample(model, prompt_ids, max_new=20, temperature=1.0, top_k=None):
        model.eval()
        out = list(prompt_ids)
        for _ in range(max_new):
            ctx = torch.tensor([out[-CONTEXT:]])
            with torch.no_grad():
                logits = model(ctx).squeeze() / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[-1]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1).item()
            out.append(next_id)
            if next_id == tok2id['</s>']:
                break
        return out

    seed = [tok2id['<s>']] * CONTEXT + [tok2id.get('the', tok2id['<s>'])]
    for temp in [0.5, 1.0, 1.5]:
        ids = sample(model, seed, max_new=15, temperature=temp)
        words = [id2tok[i] for i in ids[CONTEXT:]]
        print(f"τ={temp}: {' '.join(words)}")
else:
    print("Atlandı (PyTorch yok).")

## 6. Alıştırmalar

1. **Elle perpleksite.** $|V| = 10000$ üzerinde düzgün bir modelin herhangi bir ayrılmış metindeki perpleksitesini hesaplayın. Ardından her zaman unigram dağılımını öngören bir modelin perpleksitesini hesaplayın. Hangisi daha düşük, ne kadar?
2. **Tokenizasyondan bağımsızlık.** Perpleksitenin aynı metnin farklı tokenizasyonları altında değişmez *olmadığını* bir karşı örnekle gösterin. Tokenizer'dan bağımsız bir metrik belirtin.
3. **Hiyerarşik softmax.** 50 bin sözcüklük bir sözlük üzerindeki softmax, Bengio modelinin darboğazıdır. Hiyerarşik softmax'ı (Morin ve Bengio, 2005) — sözlük üzerinde ikili bir ağaç — uygulayın ve hızlanmayı ölçün.
4. **Ölçekleme alıştırması.** Gerçek bir derlemde (ör. Penn Treebank veya WikiText-2) Bengio dil modelini $n \in \{2, 3, 5, 10\}$ bağlam uzunlukları ve $\{32, 128, 512\}$ gizli boyutlarıyla eğitin. Test perpleksitesini parametre sayısına göre çizdirin ve aynı derlemdeki Kneser–Ney trigram modeliyle karşılaştırın. Sinirsel model hangi veri büyüklüğünden itibaren öne geçiyor? Erken durdurma ve dropout'un etkisini ölçün.
5. **`<unk>` olasılığını öğrenmek.** Eğitim derleminde yalnızca bir kez geçen sözcükleri `<unk>` ile değiştirin ve iki modelin sözlük dışı test cümlesindeki perpleksitesinin nasıl değiştiğini ölçün.

---

## Gelecek Hafta

Hafta 6 — Yinelemeli mimariler. RNN, LSTM, GRU ve BPTT — kaybolan gradyan teoreminin uygulamalı bir kanıtı dâhil.